In the policy improvement step we are given the value function and simply apply the greedy heuristic to it.

$$\pi^\prime = \mathtt{greedy}(v_\pi)$$

It can be shown that this heuristic results into a policy that is better than the one the prediction step started ($\pi^\prime \ge \pi$) and this extends into multiple iterations. We can therefore converge into an optimal policy - the interested reader can follow [this](https://www.youtube.com/watch?v=Nd1-UUMVfz4&list=PLqYmG7hTraZDM-OYHWgPebj2MfCFzFObQ&index=3) lecture for a justification. 

The step itself is a simple one-pass greedification: given $V_\pi$, compute $Q_\pi(s, a)$ for every action and take the argmax.

In [ ]:
import numpy as np

def greedify(V, P, R, gamma):
    """One-step policy improvement: π'(s) = argmax_a [ R(s,a) + γ Σ P(s'|s,a) V(s') ].

    Args:
        V: value function, shape (nS,)
        P: transition probabilities, shape (nS, nA, nS)
        R: expected immediate rewards, shape (nS, nA)
        gamma: discount factor

    Returns:
        pi: deterministic greedy policy, shape (nS,), values in [0, nA)
    """
    Q = R + gamma * P @ V              # shape (nS, nA)
    return np.argmax(Q, axis=1)

The MDP below has three states in a row and two actions: action 0 stays put, and action 1 tries to move one state to the right, succeeding with probability 0.8 and costing 0.1. Staying in state 2 earns a reward of 1 per step. You evaluate a policy exactly by solving the linear system $v_\pi = r_\pi + \gamma P_\pi v_\pi$, then call `greedify` on the result.

In [ ]:
nS, nA, gamma = 3, 2, 0.9
P = np.zeros((nS, nA, nS))
R = np.zeros((nS, nA))
for s in range(nS):
    P[s, 0, s] = 1.0                       # action 0: stay
    nxt = min(s + 1, nS - 1)
    P[s, 1, nxt] += 0.8                    # action 1: move right
    P[s, 1, s] += 0.2                      #           or slip and stay
    R[s, 1] = -0.1
R[2, 0] = 1.0                              # staying in state 2 pays

def evaluate(pi, P, R, gamma):
    """Exact policy evaluation: solve (I - γ P_π) v = r_π."""
    idx = np.arange(len(pi))
    P_pi, R_pi = P[idx, pi], R[idx, pi]
    return np.linalg.solve(np.eye(len(pi)) - gamma * P_pi, R_pi)

pi = np.array([0, 0, 0])                   # start: always stay
V = evaluate(pi, P, R, gamma)
pi_new = greedify(V, P, R, gamma)
V_new = evaluate(pi_new, P, R, gamma)

print("pi      =", pi, " v_pi      =", V.round(3))
print("pi'     =", pi_new, " v_pi'     =", V_new.round(3))
print("v_pi' >= v_pi everywhere:", bool(np.all(V_new >= V - 1e-12)))

State 1 switches to moving right because its neighbor is worth 10 under $v_\pi$. State 0 keeps staying, since under $v_\pi$ its only neighbor is worth nothing. A second round of evaluation and improvement changes state 0 as well:

In [ ]:
pi_2 = greedify(V_new, P, R, gamma)
print("pi''    =", pi_2, " v_pi''    =", evaluate(pi_2, P, R, gamma).round(3))

The full *policy iteration* algorithm alternates policy evaluation with this step until the policy is stable; see [Policy Iteration](/aiml-common/lectures/mdp/dynamic-programming-algorithms/policy-iteration) for the outer loop and a gridworld demo.

**Key references**: [@Mansour2013-kw; @Ahmed2018-yk; @Schulman2015-zd; @Huang2022-gm]